# EDA Vision Splits

Split analysis for vision datasets (deepfake and Celeb-DF).

Steps:
- Inspect processed train/val directories.
- Inspect Celeb_V2 train/val/test splits.
- Summarize class counts across splits.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from pathlib import Path
from collections import Counter

summary = {
    'processed': {},
    'celeb_v2': {},
}

processed_root = REPO_ROOT / 'data' / 'processed' / 'vision'
if processed_root.exists():
    for split in ['train', 'val']:
        split_dir = processed_root / split
        counts = {}
        for label in ['real', 'fake']:
            label_dir = split_dir / label
            if label_dir.exists():
                counts[label] = len([p for p in label_dir.rglob('*') if p.is_file()])
        summary['processed'][split] = counts
        print('Processed', split, counts)
else:
    print('Missing processed vision dir:', processed_root)

celeb_root = REPO_ROOT / 'data' / 'Celeb_V2'
if celeb_root.exists():
    for split in ['Train', 'Val', 'Test']:
        split_dir = celeb_root / split
        counts = {}
        for label in ['real', 'fake']:
            label_dir = split_dir / label
            if label_dir.exists():
                counts[label] = len([p for p in label_dir.rglob('*') if p.is_file()])
        summary['celeb_v2'][split] = counts
        print('Celeb_V2', split, counts)
else:
    print('Missing Celeb_V2 dir:', celeb_root)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_vision_splits_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize vision-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'vision' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No vision entries found in TRAINING_DATA.json')
    else:
        print('vision datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
